# CryptoVision — Coin Registry Builder

## Purpose
This notebook fetches the top 1,000 cryptocurrencies by market capitalisation
from the CoinGecko public API and persists the result as a CSV file
(`data/raw/coin_registry.csv`) consumed by the CryptoVision Streamlit dashboard.

### Why 1,000 coins?
- Covers every mainstream and mid-cap asset actively traded on major exchanges.
- Avoids defunct, rug-pulled, or illiquid tokens that appear in larger lists.
- Keeps the dropdown manageable for end users.

### When to re-run
Re-run this notebook whenever the coin ranking has materially shifted (e.g.
quarterly) or after a new high-profile token listing that you want to include.

---

## 1 · Imports & configuration

In [1]:
import os
import time
import requests
import pandas as pd
from IPython.display import display

# ── API settings ────────────────────────────────────────────────────────────
API_URL           = 'https://api.coingecko.com/api/v3/coins/markets'
TARGET_COUNT      = 1000
PER_PAGE          = 250
INTER_PAGE_SLEEP  = 1.5   # seconds (free-tier rate limit)
RATE_LIMIT_SLEEP  = 65    # seconds on HTTP 429
MAX_RETRIES       = 3

# Output path (relative to notebook location)
OUTPUT_PATH = os.path.join('..', 'raw', 'coin_registry.csv')

print(f'Target: {TARGET_COUNT} coins → {OUTPUT_PATH}')

Target: 1000 coins → ..\raw\coin_registry.csv


## 2 · Fetch coins from CoinGecko

In [2]:
def fetch_page(page_number: int) -> list[dict]:
    """Fetch one page of CoinGecko market data with retry logic."""
    params = {
        'vs_currency': 'usd',
        'order':       'market_cap_desc',
        'per_page':    PER_PAGE,
        'page':        page_number,
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(API_URL, params=params, timeout=20)
            r.raise_for_status()
            coins = r.json()
            return [
                {'ticker': f"{c['symbol'].upper()}-USD", 'name': c['name']}
                for c in coins
            ]
        except requests.HTTPError as exc:
            code = exc.response.status_code if exc.response else 0
            if code == 429:
                print(f'  ⏳ Rate limited on page {page_number}. Sleeping {RATE_LIMIT_SLEEP}s …')
                time.sleep(RATE_LIMIT_SLEEP)
            else:
                print(f'  ✗ HTTP {code} on page {page_number}: {exc}')
                return []
        except requests.RequestException as exc:
            print(f'  ✗ Request error (attempt {attempt}): {exc}')
            time.sleep(5)
    return []


def collect_coins(target: int) -> list[dict]:
    """Accumulate coin data across pages until *target* unique entries collected."""
    total_pages = -(-target // PER_PAGE)  # ceiling division
    gathered: list[dict] = []

    for page in range(1, total_pages + 1):
        print(f'  Page {page}/{total_pages} …', end=' ')
        rows = fetch_page(page)
        gathered.extend(rows)
        print(f'{len(rows)} coins  (cumulative: {len(gathered)})')

        if len(gathered) >= target:
            break
        time.sleep(INTER_PAGE_SLEEP)

    return gathered


print('Starting fetch …\n')
raw_coins = collect_coins(TARGET_COUNT)
print(f'\n→ Collected {len(raw_coins)} raw entries')

Starting fetch …

  Page 1/4 … 250 coins  (cumulative: 250)
  Page 2/4 … 250 coins  (cumulative: 500)
  Page 3/4 … 250 coins  (cumulative: 750)
  Page 4/4 …   ✗ HTTP 0 on page 4: 429 Client Error: Too Many Requests for url: https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&order=market_cap_desc&per_page=250&page=4
0 coins  (cumulative: 750)

→ Collected 750 raw entries


## 3 · Clean and deduplicate

In [3]:
# Known defunct tokens to exclude (update list as needed)
EXCLUDED_TICKERS = {
    'LUNA-USD', 'LUNC-USD', 'UST-USD', 'FTT-USD', 'SRM-USD',
    'MIR-USD', 'ANC-USD', 'TOMB-USD', 'TSHARE-USD',
}

registry_df = (
    pd.DataFrame(raw_coins)
    .drop_duplicates(subset='ticker')
    .query('ticker not in @EXCLUDED_TICKERS')
    .iloc[:TARGET_COUNT]
    .reset_index(drop=True)
)

print(f'Registry size after deduplication: {len(registry_df)} coins\n')
display(registry_df.head(10))

Registry size after deduplication: 734 coins



,ticker,name
0,BTC-USD,Bitcoin
1,ETH-USD,Ethereum
2,USDT-USD,Tether
3,XRP-USD,XRP
4,BNB-USD,BNB
5,USDC-USD,USDC
6,SOL-USD,Solana
7,TRX-USD,TRON
8,DOGE-USD,Dogecoin
9,FIGR_HELOC-USD,Figure Heloc


## 4 · Persist to CSV

In [4]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
registry_df.to_csv(OUTPUT_PATH, index=False)
print(f'✓  Saved {len(registry_df)} coins to "{os.path.abspath(OUTPUT_PATH)}"')

✓  Saved 734 coins to "C:\Users\Hussnain\Desktop\CryptoVision\data\raw\coin_registry.csv"


---

## Notes

| Topic | Detail |
|---|---|
| **API** | CoinGecko `/coins/markets` — free tier, no key required |
| **Rate limit** | ~10–30 req/min on free tier; script sleeps automatically |
| **Output** | `data/raw/coin_registry.csv` — two columns: `ticker`, `name` |
| **Ticker format** | `<SYMBOL>-USD` (e.g. `BTC-USD`) — compatible with yfinance |
| **Refresh cadence** | Quarterly recommended; or after major market events |